In [1]:
%pip install networkx matplotlib pydot

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%load_ext autoreload
%autoreload 1

import src.offline.log as cpg_parser

In [3]:
from pathlib import Path
import pydot
import networkx as nx

SERVICE_NAME = "frontend"
dot_path = Path(f"services/{SERVICE_NAME}/export.dot")

dot_text = dot_path.read_text(encoding="utf-8")
graphs = pydot.graph_from_dot_data(dot_text)

P = graphs[0]
P

In [4]:
G = nx.MultiDiGraph()

for node in P.get_nodes():
    name = node.get_name()
    if name in (None, "node", "graph", "edge"):
        continue
    nid = str(name).strip('"')
    attrs = {k: v for k, v in node.get_attributes().items()}
    G.add_node(nid, **attrs)

for edge in P.get_edges():
    src = str(edge.get_source()).strip('"')
    dst = str(edge.get_destination()).strip('"')
    attrs = {k: v for k, v in edge.get_attributes().items()}
    G.add_edge(src, dst, **attrs)

print(G.number_of_nodes(), G.number_of_edges())

30548 164385


In [5]:
templates = cpg_parser.build_templates_from_cpg(G, max_ddg_depth=5)
root = cpg_parser.build_trie(templates)
cpg_parser.visualize_trie_matplotlib(root, output_path=f"output/{SERVICE_NAME}/trie.png")

[Trie] Saved matplotlib PNG -> output/frontend/trie.png


In [6]:
print("Templates:", len(templates))
for t in templates[:10]:
    print(f"[{t.call_node_id}] {t.method_name}() -> {t.raw_template} (static={t.static_count})")

Templates: 77
[30064771307] loadDeploymentDetails() -> failed to fetch the hostname for the pod <*> (static=8)
[30064771312] loadDeploymentDetails() -> failed to fetch the name of the cluster in which the pod is running <*> (static=14)
[30064771317] loadDeploymentDetails() -> failed to fetch the zone of the node where the pod is scheduled <*> (static=13)
[30064771328] loadDeploymentDetails() -> loaded deployment details <*> (static=3)
[30064772872] AddItem() -> method additem not implemented <*> (static=4)
[30064772874] GetCart() -> method getcart not implemented <*> (static=4)
[30064772876] EmptyCart() -> method emptycart not implemented <*> (static=4)
[30064772938] ListRecommendations() -> method listrecommendations not implemented <*> (static=4)
[30064772994] ListProducts() -> method listproducts not implemented <*> (static=4)
[30064772996] GetProduct() -> method getproduct not implemented <*> (static=4)


In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# Step 1 - Collect all endpoints
from src.offline.entrypoint import EntrypointDetector, Entrypoint

print("STEP 1: COLLECTING ALL ENDPOINTS")
print("-" * 40)
detector = EntrypointDetector(G)
all_entrypoints = detector.detect()
print(f"Found {len(all_entrypoints)} entrypoints:")
for i, ep in enumerate(all_entrypoints):
    print(f"  {i+1}. {ep.name} -> {ep.full_name}")

STEP 1: COLLECTING ALL ENDPOINTS
----------------------------------------
Found 62 entrypoints:
  1. init -> main.init
  2. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.CartItem.Descriptor
  3. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.AddItemRequest.Descriptor
  4. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.EmptyCartRequest.Descriptor
  5. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.GetCartRequest.Descriptor
  6. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.Cart.Descriptor
  7. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.Empty.Descriptor
  8. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.ListRecommendationsRequest.Descriptor
  9. Descriptor -> github.com/GoogleCloudPlatform/microservices-demo/src/

In [9]:
%pip install graphviz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
all_entrypoints

[Entrypoint(node_id='107374182403', name='init', full_name='main.init', filename='deployment_details.go'),
 Entrypoint(node_id='107374182411', name='Descriptor', full_name='github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.CartItem.Descriptor', filename='genproto/demo.pb.go'),
 Entrypoint(node_id='107374182418', name='Descriptor', full_name='github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.AddItemRequest.Descriptor', filename='genproto/demo.pb.go'),
 Entrypoint(node_id='107374182425', name='Descriptor', full_name='github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.EmptyCartRequest.Descriptor', filename='genproto/demo.pb.go'),
 Entrypoint(node_id='107374182431', name='Descriptor', full_name='github.com/GoogleCloudPlatform/microservices-demo/src/frontend/genproto.GetCartRequest.Descriptor', filename='genproto/demo.pb.go'),
 Entrypoint(node_id='107374182437', name='Descriptor', full_name='github.com/GoogleCloudPlatform/micr

In [14]:
from src.offline.finite_state_machine import LogFlowExtractor
from src.offline.visual import draw_log_fsm_graphviz
extractor = LogFlowExtractor(templates)
fsms = {}
all_flows = {}
for i, ep in enumerate(all_entrypoints):
    flowresult = flowanalyzer.build(ep.node_id)
    semantic_graph = flowresult.semantic_graphs[ep.full_name]

    # image_path = draw_semantic_graph_graphviz(
    #     semantic_graph=semantic_graph,
    #     filename=f"{ep.name}",
    #     output_dir=f"output/{SERVICE_NAME}/{ep.name}/",
    #     fmt="png",
    #     rankdir="LR",
    # )

    all_flows[ep.name] = flowresult
    print(f"output/{SERVICE_NAME}/{ep.name}/")

    fsm = extractor.extract(flowresult)
    fsms[ep.name] = fsm
    # image_path = draw_log_fsm_graphviz(
    # fsm,
    #     filename=f"{ep.name}_fsm",
    #     output_dir=f"output/{SERVICE_NAME}/{ep.name}/",
    #     fmt="png",
    #     rankdir="LR",
    # )
    # print(image_path)


output/frontend/init/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/Descriptor/
output/frontend/init/
output/frontend/homeHandler/
output/frontend/productHandler/

In [15]:
from src.online.stateful_chain_store import extract_logs, logs_to_df

dataset_path="./dataset/re3ob_adservice_f3_1/logs.csv"
start_time=1731903974
end_time=1731903980
logs = extract_logs(dataset_path, SERVICE_NAME, start_time, end_time)
df_logs = logs_to_df(logs)

In [16]:
import importlib
import src.online.stateful_chain_store

importlib.reload(src.online.stateful_chain_store)

from src.online.stateful_chain_store import (
    ChainStore,
    classify_logs_with_chain_store,
)

# Выбери словарь FSM, который есть в текущем notebook.
# Если строил через build_entrypoint_fsms:
FSM_CATALOG = fsms

# Если строил через ячейку export_all:
# FSM_CATALOG = all_fsms


events_df, chains_df, chain_store = classify_logs_with_chain_store(
    logs=df_logs,
    fsms=FSM_CATALOG,
    threshold=0.55,
    time_gap_sec=30,
    max_active_chains=512,

    # Оставь None, если в dflogs есть только runtime message.
    # Если ранее template matcher записал CPG CALL id в колонку,
    # например "call_node_id", используй точное сопоставление:
    # call_node_column="call_node_id",
    call_node_column=None,
)

print("Events:", len(events_df))
print("Chains:", len(chains_df))

display(events_df.head(30))
display(chains_df.head(30))

Events: 114
Chains: 41


,log_index,timestamp,bucket,message,chain_id,entrypoint,verdict,source_states,target_states,transition_ids,templates,score
0,0,1731903974,frontend@1731903974,request started,NaN,NaN,unknown,,,,,0.00
1,1,1731903974,frontend@1731903974,serving product page,0.0,productHandler,new_chain,segment:0,segment:2 | segment:3 | segment:4 | segment:5,log_transition:0 | log_transition:1 | log_tran...,serving product page <*>,0.75
2,2,1731903974,frontend@1731903974,request complete,NaN,NaN,unknown,,,,,0.00
3,3,1731903974,frontend@1731903974,request complete,NaN,NaN,unknown,,,,,0.00
4,4,1731903974,frontend@1731903974,request started,NaN,NaN,unknown,,,,,0.00
5,5,1731903974,frontend@1731903974,adding to cart,1.0,addToCartHandler,completed,segment:0,segment:2,log_transition:0,adding to cart <*>,0.75
6,6,1731903974,frontend@1731903974,request complete,NaN,NaN,unknown,,,,,0.00
7,7,1731903974,frontend@1731903974,request started,NaN,NaN,unknown,,,,,0.00
8,8,1731903974,frontend@1731903974,view user cart,2.0,viewCartHandler,new_chain,segment:0,segment:1 | segment:2 | segment:3,log_transition:0 | log_transition:1 | log_tran...,view user cart <*>,0.75
9,9,1731903974,frontend@1731903974,request complete,NaN,NaN,unknown,,,,,0.00


,chain_id,entrypoint,status,termination_reason,created_at,last_timestamp,score,log_count,frontier_size,frontier
13,13,placeOrderHandler,active,,1731903976,1731903976,1.333333,2,2,segment:3 | segment:4
0,0,productHandler,active,,1731903974,1731903974,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
3,3,productHandler,active,,1731903975,1731903975,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
5,5,productHandler,active,,1731903975,1731903975,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
7,7,productHandler,active,,1731903976,1731903976,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
8,8,productHandler,active,,1731903976,1731903976,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
14,14,productHandler,active,,1731903976,1731903976,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
15,15,productHandler,active,,1731903976,1731903976,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
19,19,productHandler,active,,1731903978,1731903978,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5
20,20,productHandler,active,,1731903978,1731903978,0.750000,1,4,segment:2 | segment:3 | segment:4 | segment:5


In [26]:
import json
from dataclasses import asdict
from pathlib import Path

from src.offline.flow import EntrypointFlow
from src.offline.finite_state_machine import LogFlowExtractor, printlogfsm


from src.offline.flow import EntrypointFlow
from src.offline.finite_state_machine import LogFlowExtractor, printlogfsm


OUTPUT_DIR = Path("output/all_entrypoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FLOW_MAX_DEPTH = 5
RENDER_GRAPHVIZ = False  # True, если импортировал draw_log_fsm_graphviz.py


def safe_name(value: str) -> str:
    return (
        value.replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("(", "_")
        .replace(")", "_")
        .replace(" ", "_")
    )


def json_default(value):
    if hasattr(value, "value"):       # StrEnum
        return value.value
    if isinstance(value, set):
        return sorted(value)
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")


def serialize_flow(flow_result):
    return {
        "entrypoint_node_id": flow_result.entrypoint_node_id,
        "entrypoint_name": flow_result.entrypoint_name,
        "entrypoint_full_name": flow_result.entrypoint_full_name,
        "summary": flow_result.summary(),
        "cycle_warnings": flow_result.cycle_warnings,
        "max_depth_reached": flow_result.max_depth_reached,
        "external_calls": [asdict(call) for call in flow_result.external_calls],
        "methods": {
            method_name: {
                "method_node_id": semantic_graph.method_node_id,
                "start_node_id": semantic_graph.start_node_id,
                "return_node_ids": list(semantic_graph.return_node_ids),
                "input_parameters": semantic_graph.input_parameters,
                "output_parameters": semantic_graph.output_parameters,
                "nodes": [
                    {
                        "node_id": unit.node_id,
                        "kind": unit.kind,
                        "code": unit.code,
                        "line": unit.line,
                        "raw_cfg_node_ids": unit.raw_cfg_node_ids,
                        "defines": [asdict(var) for var in unit.defines],
                        "uses": [asdict(var) for var in unit.uses],
                        "call_node_ids": unit.call_node_ids,
                        "callee_full_names": unit.callee_full_names,
                        "internal_callee_full_names": unit.internal_callee_full_names,
                    }
                    for unit in semantic_graph.nodes.values()
                ],
                "edges": [asdict(edge) for edge in semantic_graph.edge_list],
            }
            for method_name, semantic_graph in flow_result.semantic_graphs.items()
        },
    }


def serialize_fsm(fsm):
    return {
        "entrypoint_node_id": fsm.entrypoint_node_id,
        "entrypoint_name": fsm.entrypoint_name,
        "entrypoint_full_name": fsm.entrypoint_full_name,
        "summary": fsm.summary(),
        "warnings": fsm.warnings,
        "start_states": sorted(fsm.start_states),
        "terminal_states": sorted(fsm.terminals),
        "states": [asdict(state) for state in fsm.states.values()],
        "transitions": [asdict(edge) for edge in fsm.transitions],
    }


flow_analyzer = EntrypointFlow(
    G,
    max_depth=FLOW_MAX_DEPTH,
)

fsm_extractor = LogFlowExtractor(templates)

all_flows = {}
all_fsms = {}
export_index = []



for index, entrypoint in enumerate(all_entrypoints, start=1):
    entrypoint_name = entrypoint.name
    artifact_name = safe_name(entrypoint_name)

    try:
        # SemanticFlowGraph для entrypoint и всех достижимых internal methods.
        flow_result = flow_analyzer.build(entrypoint.node_id)

        # Компактный FSM по SemanticFlowGraph entrypoint-а.
        fsm = fsm_extractor.extract(flow_result)

        all_flows[entrypoint_name] = flow_result
        all_fsms[entrypoint_name] = fsm

        flow_path = OUTPUT_DIR / f"{artifact_name}.flow.json"
        fsm_path = OUTPUT_DIR / f"{artifact_name}.fsm.json"

        flow_path.write_text(
            json.dumps(
                serialize_flow(flow_result),
                ensure_ascii=False,
                indent=2,
                default=json_default,
            ),
            encoding="utf-8",
        )

        fsm_path.write_text(
            json.dumps(
                serialize_fsm(fsm),
                ensure_ascii=False,
                indent=2,
                default=json_default,
            ),
            encoding="utf-8",
        )

        # Необязательная Graphviz-визуализация.
        image_path = None
        if RENDER_GRAPHVIZ:
            from draw_log_fsm_graphviz import draw_log_fsm_graphviz

            image_path = draw_log_fsm_graphviz(
                fsm,
                filename=f"{artifact_name}.fsm",
                output_dir=str(OUTPUT_DIR),
                fmt="png",
                rankdir="LR",
            )

        row = {
            "entrypoint": entrypoint_name,
            "status": "ok",
            "reachable_methods": len(flow_result.semantic_graphs),
            "semantic_nodes": sum(
                len(semantic_graph.nodes)
                for semantic_graph in flow_result.semantic_graphs.values()
            ),
            "semantic_edges": sum(
                len(semantic_graph.edges)
                for semantic_graph in flow_result.semantic_graphs.values()
            ),
            "fsm_segments": len(fsm.states),
            "fsm_transitions": len(fsm.transitions),
            "fsm_warnings": fsm.warnings,
            "flow_file": flow_path.name,
            "fsm_file": fsm_path.name,
            "fsm_image": str(image_path) if image_path else None,
        }

        export_index.append(row)

        print(
            f"[{index:>3}/{len(all_entrypoints)}] "
            f"{entrypoint_name}\n"
            f"  methods={row['reachable_methods']}, "
            f"semantic={row['semantic_nodes']}/{row['semantic_edges']}, "
            f"fsm={row['fsm_segments']}/{row['fsm_transitions']}"
        )

    except Exception as error:
        row = {
            "entrypoint": entrypoint_name,
            "status": "error",
            "error": f"{type(error).__name__}: {error}",
        }
        export_index.append(row)
        print(f"[{index:>3}/{len(all_entrypoints)}] ERROR {entrypoint_name}: {row['error']}")

(OUTPUT_DIR / "index.json").write_text(
    json.dumps(export_index, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"\nЭкспорт завершён: {OUTPUT_DIR}")
print(f"Успешно построено: {len(all_fsms)} FSM из {len(all_entrypoints)} entrypoint-ов")

[  1/62] init
  methods=1, semantic=2/2, fsm=1/0
[  2/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  3/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  4/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  5/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  6/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  7/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  8/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[  9/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 10/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 11/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 12/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 13/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 14/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 15/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 16/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 17/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 18/62] Descriptor
  methods=2, semantic=3/3, fsm=1/0
[ 19/62] Descrip